In [ ]:
import json
import pandas as pd
from reserve_ampleness.data.nyfed_load_rates import load_rates
from reserve_ampleness.data.fred_load_series import load_fred
from reserve_ampleness.data.fred_load_series import load_policy_rate


In [ ]:
df = load_rates("nyfed_all_rates", ["SOFR"])
dg = load_policy_rate()

In [ ]:
df

In [ ]:
dg

In [ ]:
dgfil = dg['2018-4-2 00:00:00':]

In [ ]:
sofrandpolicy = pd.merge(
    df, dgfil[df.index],
    left_index = True,
    right_index = True
)

In [ ]:
sofrandpolicy

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 4))
plt.plot(sofrandpolicy['SOFR_percentRate'] - sofrandpolicy['policy_rate'])

In [ ]:
from reserve_ampleness.features.calendar_features import build_calendar
from reserve_ampleness.features.calendar_features import turn_dates

cal = build_calendar(df.index)
turn = pd.to_datetime(turn_dates(cal))

sofrandpolicy = pd.merge(
    sofrandpolicy, cal,
    left_index = True,
    right_index = True
)

sofrandpolicy

In [ ]:
filtered_sofrandpolicy = (
    sofrandpolicy.groupby([sofrandpolicy.index.year, sofrandpolicy.index.month], group_keys=False).apply(lambda group: group.iloc[2:-2])
)

(filtered_sofrandpolicy['SOFR_percentPercentile99'] - filtered_sofrandpolicy['policy_rate']).resample('ME').median()

In [ ]:
per_f = filtered_sofrandpolicy.index.to_period("M")
spread = (filtered_sofrandpolicy['SOFR_percentPercentile99'] - filtered_sofrandpolicy['policy_rate'])*100
base_by_period = spread.groupby(per_f).median()

turn_per = pd.PeriodIndex(turn, freq="M")
baseline = pd.Series(base_by_period.reindex(turn_per).to_numpy(), index = turn, name='baseline_p99')
assert baseline.notna().all(), "a turn's month had no eligible baseline days"

sofrandpolicy = pd.merge(
    sofrandpolicy, baseline,
    left_index=True,
    right_index=True,
    how='outer'
)

sofrandpolicy

In [ ]:
plt.scatter(sofrandpolicy[sofrandpolicy['is_quarter_end'] == False].index, (sofrandpolicy[sofrandpolicy['is_quarter_end'] == False]['SOFR_percentPercentile99'] - sofrandpolicy[sofrandpolicy['is_quarter_end'] == False]['policy_rate'])*100 - sofrandpolicy[sofrandpolicy['is_quarter_end'] == False]['baseline_p99'])
plt.scatter(sofrandpolicy[(sofrandpolicy['is_quarter_end'] == True) & (sofrandpolicy['is_year_end'] == False)].index, (sofrandpolicy[(sofrandpolicy['is_quarter_end'] == True) & (sofrandpolicy['is_year_end'] == False)]['SOFR_percentPercentile99'] - sofrandpolicy[(sofrandpolicy['is_quarter_end'] == True) & (sofrandpolicy['is_year_end'] == False)]['policy_rate'])*100 - sofrandpolicy[(sofrandpolicy['is_quarter_end'] == True) & (sofrandpolicy['is_year_end'] == False)]['baseline_p99'])
plt.scatter(sofrandpolicy[sofrandpolicy['is_year_end'] == True].index, (sofrandpolicy[sofrandpolicy['is_year_end'] == True]['SOFR_percentPercentile99'] - sofrandpolicy[sofrandpolicy['is_year_end'] == True]['policy_rate'])*100 - sofrandpolicy[sofrandpolicy['is_year_end'] == True]['baseline_p99'])

In [ ]:
((sofrandpolicy[sofrandpolicy['is_quarter_end'] == False]['SOFR_percentPercentile99'] - sofrandpolicy[sofrandpolicy['is_quarter_end'] == False]['policy_rate'])*100 - sofrandpolicy[sofrandpolicy['is_quarter_end'] == False]['baseline_p99']).mean(), ((sofrandpolicy[(sofrandpolicy['is_quarter_end'] == True) & (sofrandpolicy['is_year_end'] == False)]['SOFR_percentPercentile99'] - sofrandpolicy[(sofrandpolicy['is_quarter_end'] == True) & (sofrandpolicy['is_year_end'] == False)]['policy_rate'])*100 - sofrandpolicy[(sofrandpolicy['is_quarter_end'] == True) & (sofrandpolicy['is_year_end'] == False)]['baseline_p99']).mean(), ((sofrandpolicy[sofrandpolicy['is_year_end'] == True]['SOFR_percentPercentile99'] - sofrandpolicy[sofrandpolicy['is_year_end'] == True]['policy_rate'])*100 - sofrandpolicy[sofrandpolicy['is_year_end'] == True]['baseline_p99']).mean()

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from reserve_ampleness.data.nyfed_load_rates import load_rates
from reserve_ampleness.data.fred_load_series import load_policy_rate
from reserve_ampleness.features.calendar_features import build_calendar, turn_dates

wide   = load_rates()
policy = load_policy_rate().reindex(wide.index).ffill()
cal    = build_calendar(wide.index)
turns  = turn_dates(cal)

def turn_type(ts):
    if ts.month == 12:        return "year_end"
    if ts.month in (3, 6, 9): return "quarter_end"
    return "month_end"

types = pd.Series([turn_type(t) for t in turns], index=turns)
OFF   = list(range(-10, 4))
pos   = {d: i for i, d in enumerate(wide.index)}

def panel(col):
    spread = (wide[col] - policy) * 100.0          # bp
    rows = {t: {k: (spread.iloc[pos[t]+k] if 0 <= pos[t]+k < len(spread) else np.nan)
                for k in OFF} for t in turns}
    return pd.DataFrame(rows).T

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True)
for ax, (col, label) in zip(axes, [("SOFR_percentRate", "headline"),
                                   ("SOFR_percentPercentile99", "p99")]):
    P = panel(col)
    med = P.groupby(types).median()
    for t in ["month_end", "quarter_end", "year_end"]:
        ax.plot(OFF, med.loc[t].values, marker="o", ms=3, label=f"{t} (n={int((types==t).sum())})")
    ax.axvline(0, lw=0.8, color="k", alpha=0.4)
    ax.set_title(f"SOFR {label} − policy, median by offset from turn")
    ax.set_xlabel("trading days from turn"); ax.set_ylabel("bp")
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

for col, label in [("SOFR_percentRate","headline"), ("SOFR_percentPercentile99","p99")]:
    print(f"\n=== {label} ===")
    print(panel(col).groupby(types).median().round(2).to_string())

In [ ]:
OFF